# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shashank007-ux/Week-1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Ranked actions + reason codes

The queue: what to do first, and why, in words a human trusts.*Pages are ranked by measured probability of the observed decline label on unseen clients. Reason codes are transparent signals for review, not explanations of causality.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists()), None)
if ROOT is None: raise FileNotFoundError('Could not find the starter dataset.')
df = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id').reset_index(drop=True)
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
feature_columns = [c for c in ['search_volume','competition','cpc','content_type','main_intent','word_count','char_count','impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d','engaged_sessions_90d','ai_sessions_90d','scroll_events_90d','days_with_impressions','days_with_sessions','impressions_prev_30d','clicks_prev_30d','sessions_prev_30d','content_age_days','age_tier_order','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct'] if c in df.columns]
categorical = [c for c in ['content_type','main_intent'] if c in feature_columns]
numeric = [c for c in feature_columns if c not in categorical]
X = pd.concat([df[numeric].apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf], np.nan).fillna(0), pd.get_dummies(df[categorical].fillna('unknown').astype(str), prefix=categorical, dtype=float)], axis=1)
clients = df['client_id'].fillna('unknown').astype(str)
rng = np.random.default_rng(RANDOM_STATE)
test_clients = set(rng.permutation(clients.drop_duplicates().to_numpy())[:max(1, int(round(clients.nunique() * .2)))])
test_mask = clients.isin(test_clients).to_numpy()
train_idx, test_idx = np.flatnonzero(~test_mask), np.flatnonzero(test_mask)
model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE).fit(X.iloc[train_idx], df['is_declining_label'].iloc[train_idx])
queue = df.iloc[test_idx].copy()
queue['decline_probability'] = model.predict_proba(X.iloc[test_idx])[:, 1]
visibility_cutoff = queue['impressions_90d'].quantile(.25)
position_cutoff = queue.loc[queue['avg_position'] > 0, 'avg_position'].quantile(.75)
age_cutoff, word_cutoff = queue['days_since_last_update'].quantile(.75), queue['word_count'].quantile(.25)
def reason_codes(row):
    reasons = []
    if row['impressions_90d'] <= visibility_cutoff: reasons.append('low_visibility')
    if row['avg_position'] == 0 or row['avg_position'] >= position_cutoff: reasons.append('weak_or_missing_position')
    if row['days_since_last_update'] >= age_cutoff: reasons.append('stale_update')
    if pd.notna(row['word_count']) and row['word_count'] <= word_cutoff: reasons.append('thin_content')
    return ';'.join(reasons) or 'model_signal_only'
queue['reason_codes'] = queue.apply(reason_codes, axis=1)
queue = queue.sort_values('decline_probability', ascending=False).reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue) + 1)
queue['recommended_action'] = np.where(queue['reason_codes'].str.contains('stale_update|thin_content'), 'human_review_refresh_candidate', 'human_review_diagnosis')
queue = queue[['rank','content_id','client_id','decline_probability','reason_codes','recommended_action','impressions_90d','avg_position','days_since_last_update','word_count']]
print(f'Client-held-out queue: {len(queue):,} rows | held-out clients: {len(test_clients)} | label base rate: {df["is_declining_label"].iloc[test_idx].mean():.3f}')
display(queue.head(10))
assert not {'trend_direction','trend_pct','content_id','client_id'}.intersection(feature_columns)

Client-held-out queue: 2,325 rows | held-out clients: 6 | label base rate: 0.391


,rank,content_id,client_id,decline_probability,reason_codes,recommended_action,impressions_90d,avg_position,days_since_last_update,word_count
0,1,content_5a10e3c2068b,client_f74efabef1,0.769609,model_signal_only,human_review_diagnosis,236,6.1,8,3267.0
1,2,content_0cf67ec37ab8,client_f74efabef1,0.763155,model_signal_only,human_review_diagnosis,767,3.0,8,3193.0
2,3,content_6e17dbac0491,client_f74efabef1,0.762851,model_signal_only,human_review_diagnosis,405,10.6,8,3844.0
3,4,content_2432fcb036bb,client_0b918943df,0.759335,model_signal_only,human_review_diagnosis,73,7.2,8,4388.0
4,5,content_f2f800ba53d5,client_f74efabef1,0.757171,model_signal_only,human_review_diagnosis,200,13.8,8,3056.0
5,6,content_071c40cf83c1,client_f74efabef1,0.754019,stale_update,human_review_refresh_candidate,1389,10.8,20,2523.0
6,7,content_575fd096bff5,client_f74efabef1,0.751056,model_signal_only,human_review_diagnosis,337,13.9,8,3896.0
7,8,content_cb3f428ab8ad,client_f74efabef1,0.750281,model_signal_only,human_review_diagnosis,450,8.5,8,3279.0
8,9,content_83be0494c955,client_f74efabef1,0.748585,model_signal_only,human_review_diagnosis,868,13.7,8,3183.0
9,10,content_7dbdac24c5ac,client_f74efabef1,0.747562,stale_update,human_review_refresh_candidate,109,8.3,20,2627.0


## 2. Intended use and limits

Editors may use the queue to choose pages for diagnosis and refresh planning. It is decision-support only: not a traffic forecast, causal estimate, quality judgment, or automatic publishing decision.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
intended_use = {'audience':'editors','purpose':'review prioritization','decision':'which pages to inspect first'}
limits = ['not causal','not a traffic forecast','not an automatic publishing decision']
print(intended_use)
print('Limits:', '; '.join(limits))
assert queue['decline_probability'].between(0, 1).all()

{'audience': 'editors', 'purpose': 'review prioritization', 'decision': 'which pages to inspect first'}
Limits: not causal; not a traffic forecast; not an automatic publishing decision


## 3. Human review + the no-go list

A person must check accuracy, search intent, evidence of decline, ownership, accessibility, and brand context. Never automate publishing, deletion, redirects, regulated advice changes, or causal conclusions from this score.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
required_review = ['accuracy','intent','ownership','accessibility','evidence of decline']
no_go = ['publish automatically','delete or redirect automatically','change regulated advice','treat score as causal evidence']
assert queue['recommended_action'].str.startswith('human_review_').all()
print('Required human checks:', ', '.join(required_review))
print('No-go actions:', ', '.join(no_go))

Required human checks: accuracy, intent, ownership, accessibility, evidence of decline
No-go actions: publish automatically, delete or redirect automatically, change regulated advice, treat score as causal evidence


## 4. Monitoring / retrain triggers

Recheck the queue after each data refresh. Retrain only when a fresh labeled window is available and the same client-held-out evaluation is rerun. Investigate material shifts in label rate, missingness, client mix, traffic scale, or top-K precision.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring = {'label_base_rate':float(df['is_declining_label'].mean()),'queue_rows':int(len(queue)),'missing_word_count_pct':float(df['word_count'].isna().mean()),'trigger':'recheck base rate, missingness, client mix, or precision@K; retrain with a fresh labeled window'}
print(json.dumps(monitoring, indent=2))
assert monitoring['queue_rows'] > 0

{
  "label_base_rate": 0.5420666666666667,
  "queue_rows": 2325,
  "missing_word_count_pct": 0.2566333333333333,
  "trigger": "recheck base rate, missingness, client mix, or precision@K; retrain with a fresh labeled window"
}


## 5. Exports for the paper

The following cell writes the ranked queue and monitoring summary to `work/outputs/`.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
output_dir = ROOT / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / 'content_action_queue.csv'
monitor_path = output_dir / 'content_action_monitoring.json'
queue.to_csv(queue_path, index=False)
monitor_path.write_text(json.dumps(monitoring, indent=2), encoding='utf-8')
print('Wrote:', queue_path)
print('Wrote:', monitor_path)
assert queue_path.exists() and monitor_path.exists()

Wrote: /content/flyrank-ml-internship-starter/work/outputs/content_action_queue.csv
Wrote: /content/flyrank-ml-internship-starter/work/outputs/content_action_monitoring.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.